In [3]:
import os
import pandas as pd
import numpy as np

# Asegurar carpeta de salida
os.makedirs("dataset/processed", exist_ok=True)

# 1) Cargar
sales = pd.read_csv("dataset/sales_train_validation.csv")
sales_eval = pd.read_csv("dataset/sales_train_evaluation.csv")  # ground truth real de d_1914..d_1941
cal = pd.read_csv("dataset/calendar.csv", parse_dates=["date"])
prices = pd.read_csv("dataset/sell_prices.csv")

# 2) Wide -> long para contexto histórico (d_1..d_1913)
day_cols = [c for c in sales.columns if c.startswith("d_")]
long = sales.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=day_cols,
    var_name="d",
    value_name="target",
)

# 3) Merge con calendar
long = long.merge(cal, on="d", how="left")

# 4) Merge con precios
long = long.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# 5) ID limpio para Chronos-2
long["id"] = long["item_id"].astype(str) + "_" + long["store_id"].astype(str)
long["timestamp"] = pd.to_datetime(long["date"])

# 6) SNAP unificado
long["snap"] = np.select(
    [
        long["state_id"].eq("CA"),
        long["state_id"].eq("TX"),
        long["state_id"].eq("WI"),
    ],
    [
        long["snap_CA"],
        long["snap_TX"],
        long["snap_WI"],
    ],
    default=np.nan,
)

# 7) Eventos simplificados
long["event_any"] = (
        long["event_name_1"].notna() | long["event_name_2"].notna()
).astype(int)

long["event_type_main"] = long["event_type_1"].fillna(
    long["event_type_2"]
).fillna("None")

# 8) Selección de columnas para contexto
context_cols = [
    "id", "timestamp", "target",
    "sell_price", "snap", "wday", "month",
    "event_any", "event_type_main"
]

context_df = long[context_cols].sort_values(["id", "timestamp"]).copy()

# 9) Guardar histórico
context_df.to_csv("dataset/processed/m5_context.csv", index=False)

# 10) Construir future_df para d_1914..d_1941
future_cal = cal.iloc[1913:1941].copy()  # 28 días futuros respecto a validation
future_days = future_cal["d"].tolist()

base_ids = sales[["item_id", "store_id", "state_id"]].copy()
future_df = base_ids.merge(future_cal, how="cross")
future_df = future_df.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

future_df["id"] = future_df["item_id"].astype(str) + "_" + future_df["store_id"].astype(str)
future_df["timestamp"] = pd.to_datetime(future_df["date"])

future_df["snap"] = np.select(
    [
        future_df["state_id"].eq("CA"),
        future_df["state_id"].eq("TX"),
        future_df["state_id"].eq("WI"),
    ],
    [
        future_df["snap_CA"],
        future_df["snap_TX"],
        future_df["snap_WI"],
    ],
    default=np.nan,
)

future_df["event_any"] = (
        future_df["event_name_1"].notna() | future_df["event_name_2"].notna()
).astype(int)

future_df["event_type_main"] = future_df["event_type_1"].fillna(
    future_df["event_type_2"]
).fillna("None")

future_df = future_df[
    ["id", "timestamp", "sell_price", "snap", "wday", "month", "event_any", "event_type_main"]
].sort_values(["id", "timestamp"])

future_df.to_csv("dataset/processed/m5_future.csv", index=False)

# 11) Generar m5_actuals.csv con la verdad real de d_1914..d_1941
actuals_long = sales_eval.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=future_days,
    var_name="d",
    value_name="target",
)

actuals_long = actuals_long.merge(
    cal[["d", "date"]],
    on="d",
    how="left"
)

actuals_long["id"] = actuals_long["item_id"].astype(str) + "_" + actuals_long["store_id"].astype(str)
actuals_long["timestamp"] = pd.to_datetime(actuals_long["date"])

actuals_df = actuals_long[
    ["id", "timestamp", "target"]
].sort_values(["id", "timestamp"]).copy()

actuals_df.to_csv("dataset/processed/m5_actuals.csv", index=False)

print("Ficheros generados correctamente:")
print("- dataset/processed/m5_context.csv")
print("- dataset/processed/m5_future.csv")
print("- dataset/processed/m5_actuals.csv")
print()
print("Shapes:")
print("context_df:", context_df.shape)
print("future_df :", future_df.shape)
print("actuals_df:", actuals_df.shape)

Ficheros generados correctamente:
- dataset/processed/m5_context.csv
- dataset/processed/m5_future.csv
- dataset/processed/m5_actuals.csv

Shapes:
context_df: (58327370, 9)
future_df : (853720, 8)
actuals_df: (853720, 3)


In [4]:
# Comprobación rápida de alineación entre future y actuals

future_df = pd.read_csv("dataset/processed/m5_future.csv", parse_dates=["timestamp"])
actuals_df = pd.read_csv("dataset/processed/m5_actuals.csv", parse_dates=["timestamp"])

print("Filas future :", len(future_df))
print("Filas actuals:", len(actuals_df))

# Deben coincidir exactamente en id y timestamp
key_future = future_df[["id", "timestamp"]].drop_duplicates().sort_values(["id", "timestamp"]).reset_index(drop=True)
key_actuals = actuals_df[["id", "timestamp"]].drop_duplicates().sort_values(["id", "timestamp"]).reset_index(drop=True)

print("Mismo número de claves:", len(key_future) == len(key_actuals))
print("Claves idénticas      :", key_future.equals(key_actuals))

# Vista rápida
display(actuals_df.head())

Filas future : 853720
Filas actuals: 853720
Mismo número de claves: True
Claves idénticas      : True


,id,timestamp,target
0,FOODS_1_001_CA_1,2016-04-25,2
1,FOODS_1_001_CA_1,2016-04-26,0
2,FOODS_1_001_CA_1,2016-04-27,0
3,FOODS_1_001_CA_1,2016-04-28,0
4,FOODS_1_001_CA_1,2016-04-29,0
